In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "pinn_swe_reflections").is_dir():
            return candidate
    raise RuntimeError("Cannot find project root. Run this notebook from inside the repository.")


ROOT_DIR = find_project_root()
SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pinn_swe_reflections.models import (  # noqa: E402
    GradientEnhancedPINN,
    PINN,
    ResidualAdaptiveDistributionPINN,
    ResidualAdaptiveRefinementDistributionPINN,
)

TRAINING_RESULTS_DIR = Path(
    os.environ.get("PINN_SWE_TRAINING_RESULTS_DIR", ROOT_DIR / "training_results")
).expanduser().resolve()
OUTPUT_DIR = ROOT_DIR / "analysis_outputs" / "best_state_dict_evaluation"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_NUMERICAL_SOLUTION_DIRECTORY = "AH=5e+4"
PREDICTION_BATCH_SIZE = 65536
SAVE_OUTPUTS = True
SAVE_PREDICTIONS = True
FIG_DPI = 180

CHECKPOINT_EXPERIMENTS = [
    {
        "name": "baseline",
        "label": "PINN baseline",
        "run_dir": TRAINING_RESULTS_DIR / "baseline",
        "model_number": 0,
        # "checkpoint": TRAINING_RESULTS_DIR / "baseline" / "Best_state_dict_0.pt",
        # "numerical_solution_directory": "AH=5e+4",
    },
    # {
    #     "name": "gpinn_w001",
    #     "label": "gPINN w=0.01",
    #     "run_dir": TRAINING_RESULTS_DIR / "gpinn_w001",
    #     "model_number": 0,
    # },
]

if SAVE_OUTPUTS:
    (OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}")
print(f"Training results: {TRAINING_RESULTS_DIR}")
print(f"Device: {DEVICE}")

In [ ]:
MODEL_CLASSES = {
    "PINN": PINN,
    "GradientEnhancedPINN": GradientEnhancedPINN,
    "ResidualAdaptiveDistributionPINN": ResidualAdaptiveDistributionPINN,
    "ResidualAdaptiveRefinementDistributionPINN": ResidualAdaptiveRefinementDistributionPINN,
}

VARIABLE_SPECS = {
    "eta": {"label": "eta / zeta", "unit": "m"},
    "u": {"label": "u", "unit": "m/s"},
}


def read_json(path, default=None):
    path = Path(path)
    if not path.is_file():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def optimizer_from_string(text):
    text = str(text)
    if "Adam" in text or "adam" in text:
        return torch.optim.Adam
    if "LBFGS" in text or "lbfgs" in text:
        return torch.optim.LBFGS
    warnings.warn(f"Unknown optimizer in hyperparameters: {text}. Falling back to Adam for reconstruction.")
    return torch.optim.Adam


def activation_from_string(text):
    text = str(text).lower()
    if "tanh" in text:
        return torch.tanh
    if "sin" in text:
        return torch.sin
    if "relu" in text:
        return torch.relu
    warnings.warn(f"Unknown activation in hyperparameters: {text}. Falling back to torch.tanh.")
    return torch.tanh


def checkpoint_candidates(run_dir, model_number):
    run_dir = Path(run_dir)
    candidates = [
        run_dir / f"Best_state_dict_{model_number}.pt",
        run_dir / "Best_state_dict.pt",
    ]
    return candidates


def resolve_checkpoint(exp):
    if exp.get("checkpoint") is not None:
        checkpoint = Path(exp["checkpoint"]).expanduser().resolve()
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        return checkpoint

    run_dir = Path(exp["run_dir"]).expanduser().resolve()
    model_number = int(exp.get("model_number", 0))
    for candidate in checkpoint_candidates(run_dir, model_number):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"No checkpoint found for {exp.get('name', run_dir.name)}. Tried: "
        + ", ".join(str(p.name) for p in checkpoint_candidates(run_dir, model_number))
    )


def state_dict_from_checkpoint(path):
    obj = torch.load(path, map_location="cpu")
    if obj is None:
        raise ValueError(f"Checkpoint contains None: {path}")
    if isinstance(obj, dict) and "state_dict" in obj:
        obj = obj["state_dict"]
    if not isinstance(obj, dict):
        raise TypeError(f"Unsupported checkpoint object type {type(obj)} in {path}")
    return obj


def model_init_kwargs(hp, exp):
    model_number = int(exp.get("model_number", hp.get("model_number", 0) or 0))
    number_of_models = int(hp.get("number_of_models", 1) or 1)
    fraction = [model_number / number_of_models, (model_number + 1) / number_of_models]
    number_of_layers = int(hp.get("number_of_layers", 4))
    neurons_per_layer = int(hp.get("neurons_per_layer", 100))
    layer_sizes = [2] + number_of_layers * [neurons_per_layer] + [2]

    kwargs = {
        "layer_sizes": layer_sizes,
        "activation_function": activation_from_string(hp.get("activation_function", "tanh")),
        "optimizer": optimizer_from_string(hp.get("optimizer", "Adam")),
        "learning_rate": float(hp.get("learning_rate", 5e-4)),
        "line_search": hp.get("line_search"),
        "boundary_condition_weight": float(hp.get("boundary_condition_weight", 1.0)),
        "initial_condition_weight": float(hp.get("initial_condition_weight", 1.0)),
        "symbolic_function_weight": float(hp.get("symbolic_function_weight", 1.0)),
        # Small datasets are enough for reconstruction; evaluation uses reference grids below.
        "boundary_condition_batch_size": int(exp.get("eval_boundary_condition_batch_size", 16)),
        "initial_condition_batch_size": int(exp.get("eval_initial_condition_batch_size", 16)),
        "symbolic_function_batch_size": int(exp.get("eval_symbolic_function_batch_size", 16)),
        "training_steps": 1,
        "batch_resampling_period": int(hp.get("batch_resampling_period", 1) or 1),
        "console_output_period": int(hp.get("console_output_period", 1) or 1),
        "device": DEVICE,
        "gravitational_acceleration": float(hp.get("gravitational_acceleration", 9.81)),
        "average_sea_level": float(hp.get("average_sea_level", 100.0)),
        "momentum_dissipation": float(hp.get("momentum_dissipation", 0.0)),
        "nonlinear_drag_coefficient": float(hp.get("nonlinear_drag_coefficient", 0.0)),
        "initial_perturbation_amplitude": float(hp.get("initial_perturbation_amplitude", 1.0)),
        "non_dimensionalization": bool(hp.get("non_dimensionalization", True)),
        "vertical_length_scale": float(hp.get("vertical_length_scale", 1.0)),
        "vertical_scaling_factor": float(hp.get("vertical_scaling_factor", 1.0)),
        "horizontal_length_scale": float(hp.get("horizontal_length_scale", 1e6)),
        "time_scale": float(hp.get("time_scale", 86400.0)),
        "minimum_time": float(hp.get("minimum_time", 0.0)),
        "maximum_time": float(hp.get("maximum_time", 1.0)),
        "minimum_x": float(hp.get("minimum_x", -1.0)),
        "maximum_x": float(hp.get("maximum_x", 1.0)),
        "projected_gradients": bool(hp.get("projected_gradients", False)),
        "save_output_over_training": False,
        "save_symbolic_function_over_training": False,
        "numerical_solution_time_interval": hp.get("numerical_solution_time_interval", [0.0, 270000.0]),
        "numerical_solution_time_step": float(hp.get("numerical_solution_time_step", 60.0)),
        "numerical_solution_x_interval": hp.get("numerical_solution_x_interval", [-1000000.0, 1000000.0]),
        "numerical_solution_space_step": float(hp.get("numerical_solution_space_step", 10000.0)),
        "fraction_of_time_interval": fraction,
        "model_number": model_number,
        "number_of_models": number_of_models,
        "train_on_solution": bool(hp.get("train_on_solution", False)),
        "train_on_PINNs_Loss": bool(hp.get("train_on_PINNs_Loss", True)),
        "boundary_condition_transition_function": bool(hp.get("boundary_condition_transition_function", False)),
        "initial_condition_transition_function": bool(hp.get("initial_condition_transition_function", False)),
        "split_networks": bool(hp.get("split_networks", False)),
        "train_on_boundary_condition_loss": bool(hp.get("train_on_boundary_condition_loss", True)),
        "train_on_initial_condition_loss": bool(hp.get("train_on_initial_condition_loss", True)),
        "momentum_advection": bool(hp.get("momentum_advection", True)),
        "mixed_activation_functions": bool(hp.get("mixed_activation_functions", False)),
        "sirens_initialization": bool(hp.get("sirens_initialization", False)),
        "learning_rate_annealing": bool(hp.get("learning_rate_annealing", False)),
        "minibatch_training": bool(hp.get("minibatch_training", True)),
        "pde_mini_batch_size": int(exp.get("eval_pde_mini_batch_size", 16)),
        "bc_mini_batch_size": int(exp.get("eval_bc_mini_batch_size", 16)),
        "ic_mini_batch_size": int(exp.get("eval_ic_mini_batch_size", 16)),
        "iterations_per_epoch": 1,
        "numerical_solution_directory": exp.get(
            "numerical_solution_directory",
            hp.get("numerical_solution_directory", DEFAULT_NUMERICAL_SOLUTION_DIRECTORY),
        ),
    }

    kwargs.update(hp.get("model_kwargs", {}) or {})
    kwargs.update(exp.get("model_kwargs", {}) or {})
    return kwargs


def compute_metrics(reference, prediction):
    reference = np.asarray(reference, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    diff = prediction - reference
    l1_denom = np.sum(np.abs(reference))
    l2_denom = np.linalg.norm(reference.ravel())
    return {
        "relative_l1": np.nan if l1_denom == 0 else np.sum(np.abs(diff)) / l1_denom,
        "relative_l2": np.nan if l2_denom == 0 else np.linalg.norm(diff.ravel()) / l2_denom,
        "mse": np.mean(diff ** 2),
        "rmse": np.sqrt(np.mean(diff ** 2)),
        "mae": np.mean(np.abs(diff)),
        "max_abs": np.max(np.abs(diff)),
        "n_points": int(diff.size),
    }


def safe_name(text):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(text))


def save_figure(fig, stem):
    if not SAVE_OUTPUTS:
        return
    fig.savefig(OUTPUT_DIR / "figures" / f"{stem}.png", dpi=FIG_DPI, bbox_inches="tight")


In [ ]:
def batched_forward(model, t, x, output_column, batch_size=PREDICTION_BATCH_SIZE):
    outputs = []
    n = t.shape[0]
    model.eval()
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            out = model(t[start:end], x[start:end])[:, output_column:output_column + 1]
            outputs.append(out.detach().cpu())
    return torch.cat(outputs, dim=0)


def evaluate_model_on_reference_grid(model):
    eta_pred = batched_forward(
        model,
        model.zeta_solution_time_input_grid,
        model.zeta_solution_x_input_grid,
        output_column=1,
    ).numpy().reshape(model.zeta_solution_mesh_grid_shape)
    u_pred = batched_forward(
        model,
        model.u_solution_time_input_grid,
        model.u_solution_x_input_grid,
        output_column=0,
    ).numpy().reshape(model.u_solution_mesh_grid_shape)

    if model.non_dimensionalization:
        eta_pred = eta_pred * model.vertical_length_scale
        u_pred = u_pred * model.horizontal_length_scale / model.time_scale

    eta_ref = model.exact_solution_h_values.detach().cpu().numpy()
    u_ref = model.exact_solution_u_values.detach().cpu().numpy()

    return {
        "eta": {
            "reference": eta_ref,
            "prediction": eta_pred,
            "abs_error": np.abs(eta_pred - eta_ref),
            "time_mesh": model.dimensional_zeta_solution_time_mesh_grid,
            "x_mesh": model.dimensional_zeta_solution_x_mesh_grid,
        },
        "u": {
            "reference": u_ref,
            "prediction": u_pred,
            "abs_error": np.abs(u_pred - u_ref),
            "time_mesh": model.dimensional_u_solution_time_mesh_grid,
            "x_mesh": model.dimensional_u_solution_x_mesh_grid,
        },
    }


def plot_field_triptych(run, variable):
    spec = VARIABLE_SPECS[variable]
    field = run["fields"][variable]
    reference = field["reference"]
    prediction = field["prediction"]
    abs_error = field["abs_error"]
    time_mesh = np.asarray(field["time_mesh"], dtype=np.float64) / 86400.0
    x_mesh = np.asarray(field["x_mesh"], dtype=np.float64) / 1000.0

    vmax = max(np.nanmax(np.abs(reference)), np.nanmax(np.abs(prediction)))
    error_vmax = np.nanmax(abs_error)
    panels = [
        ("reference", reference, "RdBu_r", -vmax, vmax),
        ("checkpoint prediction", prediction, "RdBu_r", -vmax, vmax),
        ("absolute error", abs_error, "magma", 0.0, error_vmax),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
    for ax, (title, values, cmap, vmin, vmax_panel) in zip(axes, panels):
        mesh = ax.pcolormesh(time_mesh, x_mesh, values, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax_panel, rasterized=True)
        ax.set_title(title)
        ax.set_xlabel("time, days")
        ax.set_ylabel("x, km")
        fig.colorbar(mesh, ax=ax, label=spec["unit"])

    fig.suptitle(f"{run['label']} - {spec['label']}", y=1.05)
    save_figure(fig, f"checkpoint_field_triptych_{safe_name(run['name'])}_{variable}")
    return fig

In [ ]:
evaluated_runs = []
metric_rows = []

for exp in CHECKPOINT_EXPERIMENTS:
    run_dir = Path(exp["run_dir"]).expanduser().resolve()
    hp = read_json(run_dir / "Hyper_Parameter_Dictionary.json", default={})
    checkpoint = resolve_checkpoint(exp)
    model_class_name = exp.get("model_class", hp.get("model_class", "PINN"))
    model_cls = MODEL_CLASSES[model_class_name]

    print(f"Evaluating {exp.get('name', run_dir.name)} from {checkpoint.name}")
    model = model_cls(**model_init_kwargs(hp, exp)).to(DEVICE)
    state_dict = state_dict_from_checkpoint(checkpoint)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        warnings.warn(f"{exp.get('name', run_dir.name)} load_state_dict: missing={missing}, unexpected={unexpected}")
    model.to(DEVICE)
    model.eval()

    fields = evaluate_model_on_reference_grid(model)
    run = {
        "name": exp.get("name", run_dir.name),
        "label": exp.get("label", exp.get("name", run_dir.name)),
        "run_dir": run_dir,
        "checkpoint": checkpoint,
        "hyperparameters": hp,
        "fields": fields,
    }

    for variable, field in fields.items():
        row = {
            "experiment": run["name"],
            "label": run["label"],
            "variable": variable,
            "checkpoint": str(checkpoint),
            "model_class": model_class_name,
            "best_step": hp.get("best_step"),
            "epochs": hp.get("epochs"),
            "computation_time_s": hp.get("computation_time"),
        }
        row.update(compute_metrics(field["reference"], field["prediction"]))
        metric_rows.append(row)

    if SAVE_OUTPUTS and SAVE_PREDICTIONS:
        pred_dir = OUTPUT_DIR / "predictions" / safe_name(run["name"])
        pred_dir.mkdir(parents=True, exist_ok=True)
        for variable, field in fields.items():
            np.save(pred_dir / f"{variable}_reference.npy", field["reference"])
            np.save(pred_dir / f"{variable}_prediction_from_checkpoint.npy", field["prediction"])
            np.save(pred_dir / f"{variable}_abs_error_from_checkpoint.npy", field["abs_error"])

    evaluated_runs.append(run)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(metric_rows)
summary_columns = [
    "experiment",
    "label",
    "variable",
    "relative_l1",
    "relative_l2",
    "mse",
    "rmse",
    "mae",
    "max_abs",
    "best_step",
    "epochs",
    "checkpoint",
]
summary_df = metrics_df[summary_columns].sort_values(["variable", "relative_l2", "experiment"])
display(summary_df)

if SAVE_OUTPUTS:
    tables_dir = OUTPUT_DIR / "tables"
    summary_df.to_csv(tables_dir / "checkpoint_final_metrics.csv", index=False)
    try:
        (tables_dir / "checkpoint_final_metrics.tex").write_text(summary_df.to_latex(index=False), encoding="utf-8")
    except Exception as exc:
        warnings.warn(f"Could not write LaTeX table: {exc}")

In [ ]:
metrics_to_plot = ["relative_l2", "relative_l1", "rmse"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5.2 * len(metrics_to_plot), 4.2), squeeze=False)

for ax, metric in zip(axes.ravel(), metrics_to_plot):
    table = summary_df.pivot(index="label", columns="variable", values=metric)
    table.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=30)

fig.tight_layout()
save_figure(fig, "checkpoint_final_metric_bars")
plt.show()

In [ ]:
for run in evaluated_runs:
    for variable in VARIABLE_SPECS:
        fig = plot_field_triptych(run, variable)
        plt.show()